In [ ]:
import geopandas as gpd
import time
import folium
from pathlib import Path

#This is purely for performance measurement and has no effect on the actual map generation.
#Lines that are used to measure performance will be marked with comments.
start_total = time.perf_counter()

# This gets the folder where the script is located, which is assumed to be the "codes" folder in the project structure.
# Made this to work with both Jupyter Notebook and standard Python environments, as __file__ is not defined in Jupyter Notebooks.
try:
    codes_dir = Path(__file__).resolve().parent
except NameError:
    codes_dir = Path.cwd()

# project root directory (one level up from the codes directory)
project_root = codes_dir.parent

# ------Create an output folder if it doesn't exist to save the results.------
output_folder = project_root / "output"
output_folder.mkdir(exist_ok=True)

# --- Load the GeoPackage dynamically from the output folder where it was saved by the data processing script ---
gpkg_path = output_folder / "road_condition.gpkg"
start = time.perf_counter() #<-- Performance measurement
gdf = gpd.read_file(gpkg_path)
print(f"Read time: {time.perf_counter() - start:.3f} s") #<-- Performance measurement

# --- Read the GeoPackage and ensure it's in WGS84 (EPSG:4326) for compatibility with web mapping ---
gdf = gdf.to_crs(epsg=4326)

# --- Drop rows without geometry ---
gdf = gdf[gdf.geometry.notnull()].copy()

# --- Create a simplified web copy for faster rendering ---
start = time.perf_counter() #<-- Performance measurement
gdf_web = gdf.to_crs(epsg=3857).copy()
gdf_web["geometry"] = gdf_web.geometry.simplify(5, preserve_topology=True)
gdf_web = gdf_web.to_crs(epsg=4326)
print(f"Simplify time: {time.perf_counter() - start:.3f} s") #<-- Performance measurement

# --- drop empty geometries created by simplification ---
gdf_web = gdf_web[gdf_web.geometry.notnull() & ~gdf_web.geometry.is_empty].copy()

# --- Choose a center point for the map ---
finland_bounds = [
    [59.5, 19.0], # Southwest: latitude, longitude
    [70.5, 32.5]  # Northeast: latitude, longitude
]

#----------create the map with openstreetmap tiles and set the initial view to the center of the data---------
Map = folium.Map(
    location=[64.5, 26.0],
    zoom_start=5,
    tiles="OpenStreetMap",
    max_bounds=True
)

#-----limit the map to the bounds of Finland for better user experience and tiny performance boost-----
Map.fit_bounds(finland_bounds) #<-- DOES NOT WORK YET BUT DOES COMPILE. PLS DON'T TOUCH THIS FOR NOW. I WILL FIX IT LATER.

# --- Style each road using the stored color_hex field ---
def style_function(feature):
    color = feature["properties"].get("color_hex", "#808080")
    return {
        "color": color,
        "weight": 3,
        "opacity": 0.8
    }

# --- Tooltip fields ---
tooltip_fields = [col for col in ["Tie", "lane_index", "RMS_yhd", "color_hex"] if col in gdf.columns]
tooltip_aliases = tooltip_fields.copy()

tooltip = folium.GeoJsonTooltip(
    fields=tooltip_fields,
    aliases=tooltip_aliases,
    localize=True,
)

# --- Add roads to the map ---
folium.GeoJson(
    gdf_web,
    style_function=style_function,
    tooltip=tooltip,
    name="Road condition"
).add_to(Map)

# --- Set map bounds to Finland ---
Map.options["min_lat"] = 59.5
Map.options["max_lat"] = 70.5
Map.options["min_lon"] = 19.0
Map.options["max_lon"] = 32.5

# --- Add layer control to toggle the road condition layer ----
# Shows on the right top corner of the map
folium.LayerControl().add_to(Map)

# --- Save to HTML ------
start = time.perf_counter() #<-- Performance measurement
Map.save(output_folder / "road_condition_map.html")
print(f"HTML save time: {time.perf_counter() - start:.3f} s") #<-- Performance measurement
print("Map saved as road_condition_map.html")

print(f"Total runtime: {time.perf_counter() - start_total:.3f} s") #<-- Performance measurement